In [ ]:
import sys
import json
import pandas as pd
from pathlib import Path

# Connect to src directory
ml_root = Path().resolve().parents[0]
sys.path.append(str(ml_root))

from src.preprocessing import preprocess_training_data
from src.inference import RULInferencePipeline
from src.config import DATA_DIR

print("Loading Unseen Test Data...")
# Load the raw test data (using the exact same column names from Day 1)
columns = ["engine_id", "cycle", "op_setting_1", "op_setting_2", "op_setting_3"] + [f"sensor_{i}" for i in range(1, 22)]
raw_test = pd.read_csv(DATA_DIR / "raw" / "test_FD001.txt", sep="\s+", names=columns)

# Run it through our Day 7 preprocessing pipeline
processed_test = preprocess_training_data(raw_test)

print("Initializing Inference Pipeline...")
pipeline = RULInferencePipeline()
print("Pipeline Ready!")

In [ ]:
print("=" * 50)
print("BATCH INFERENCE TEST")
print("=" * 50)

# Test the first 3 engines in the dataset
for engine_id in processed_test['engine_id'].unique()[:3]:
    engine_data = processed_test[processed_test['engine_id'] == engine_id]
    
    result = pipeline.predict(engine_data)
    
    print(f"\nENGINE {engine_id} STATUS:")
    print(f"RUL    : {result['predicted_rul']} Flights")
    print(f"Health : {result['health_score']}%")
    print(f"Risk   : {result['risk_level']}")
    print(f"Action : {result['recommendation']}")

In [ ]:
# Extract full JSON for Engine 1
engine_1_data = processed_test[processed_test['engine_id'] == 1]
final_json = pipeline.predict(engine_1_data)

# Save to docs folder
docs_path = ml_root.parent / "docs"
docs_path.mkdir(parents=True, exist_ok=True)
sample_path = docs_path / "sample_prediction.json"

with open(sample_path, "w") as f:
    json.dump(final_json, f, indent=4)

print("=" * 60)
print("Complete JSON Payload structure:")
print(json.dumps(final_json, indent=4))
print(f"\nSaved successfully to {sample_path}")